# Treino e avaliação da MobileNetV2

Notebook de execução para Google Colab. Configure o runtime com GPU em **Runtime → Change runtime type → T4 GPU**. O projeto e o ZIP do TrashNet precisam estar no Google Drive; o dataset será copiado para o disco temporário da sessão para acelerar a leitura.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Ajuste estes caminhos para os locais usados no seu Google Drive.
PROJECT_DIR = Path("/content/drive/MyDrive/EcoPlay/iot-ai")
DATA_ZIP = Path("/content/drive/MyDrive/EcoPlay/trashnet.zip")
LOCAL_ZIP = Path("/content/trashnet.zip")
EXTRACT_DIR = Path("/content/trashnet")
OUTPUT_DIR = PROJECT_DIR / "models"

assert (PROJECT_DIR / "scripts/train_classifier.py").is_file(), f"Projeto não encontrado: {PROJECT_DIR}"
assert DATA_ZIP.is_file(), f"ZIP não encontrado: {DATA_ZIP}"

## Dependências e dispositivo

O treino usa PyTorch, torchvision e MobileNetV2 pré-treinada no ImageNet. A primeira execução precisa baixar os pesos pré-treinados.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-train.txt")],
    check=True,
)
import torch
print("PyTorch:", torch.__version__)
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Extrair e localizar as classes

O carregador procura a pasta que contém as seis subpastas de classe, então a estrutura interna do ZIP pode ter níveis extras.

In [ ]:
import shutil
import zipfile

shutil.copy2(DATA_ZIP, LOCAL_ZIP)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as arquivo_zip:
    arquivo_zip.extractall(EXTRACT_DIR)

classes_esperadas = {"cardboard", "glass", "metal", "paper", "plastic", "trash"}
candidatos = [
    pasta.parent
    for pasta in EXTRACT_DIR.rglob("cardboard")
    if pasta.is_dir()
    and classes_esperadas.issubset({item.name for item in pasta.parent.iterdir() if item.is_dir()})
]
if not candidatos:
    raise FileNotFoundError("Não encontrei uma pasta com as seis classes do TrashNet no ZIP.")
DATA_DIR = candidatos[0]
print("Dataset:", DATA_DIR)
print("Classes:", sorted(item.name for item in DATA_DIR.iterdir() if item.is_dir()))

## Treinar, avaliar e exportar

A separação é estratificada em treino, validação e teste. O relatório no teste TrashNet é mantido separado da avaliação opcional em fotos reais. Para esta última, crie no Drive `data/raw/real/<classe>/` com imagens próprias rotuladas e não usadas durante o treino.

In [ ]:
comando = [
    sys.executable,
    str(PROJECT_DIR / "scripts/train_classifier.py"),
    "--data", str(DATA_DIR),
    "--output", str(OUTPUT_DIR),
    "--epochs", "10",
]
REAL_DATA_DIR = PROJECT_DIR / "data/raw/real"
if REAL_DATA_DIR.is_dir():
    comando += ["--real-data", str(REAL_DATA_DIR)]
subprocess.run(comando, cwd=PROJECT_DIR, check=True)

In [ ]:
import json

relatorio = json.loads((OUTPUT_DIR / "evaluation.json").read_text(encoding="utf-8"))
print("Tamanhos das divisões:", relatorio["split_sizes"])
print("Acurácia no teste TrashNet:", round(relatorio["test"]["accuracy"], 4))
print("Macro F1 no teste TrashNet:", round(relatorio["test"]["classification_report"]["macro avg"]["f1-score"], 4))
print("Avaliação em fotos reais:", relatorio["real_photos"]["status"])
print("Arquivos exportados:", sorted(p.name for p in OUTPUT_DIR.iterdir()))

## Interpretação

Registre os valores produzidos pelo treino, sem estimá-los antes. O TrashNet tem imagens em cenários controlados; métricas nesse conjunto não demonstram, por si só, o desempenho em fotos reais. Os artefatos `mobilenetv2.onnx` e `metadata.json` são necessários para executar a API FastAPI localmente.